# Step 1: Download CMS DE-SynPUF Sample 1
Synthetic Medicare claims, 2008-2010.

In [10]:
from pathlib import Path
import requests, zipfile

BASE = Path.home() / "Desktop/projects/claims_analysis"
RAW = BASE / "data/raw"
RAW.mkdir(parents=True, exist_ok=True)
for d in ["sql", "notebooks", "outputs", "powerbi"]:
    (BASE / d).mkdir(exist_ok=True)

In [13]:
SAMPLE = 2   # Sample 1 is missing its 2010 member file on the CMS site
s = SAMPLE

cms = "https://www.cms.gov/research-statistics-data-and-systems/downloadable-public-use-files/synpufs/downloads/"
dl = "http://downloads.cms.gov/files/"

URLS = {
    f"bene_2008_s{s}":  cms + f"de1_0_2008_beneficiary_summary_file_sample_{s}.zip",
    f"bene_2009_s{s}":  cms + f"de1_0_2009_beneficiary_summary_file_sample_{s}.zip",
    f"bene_2010_s{s}":  cms + f"de1_0_2010_beneficiary_summary_file_sample_{s}.zip",
    f"inpatient_s{s}":  cms + f"de1_0_2008_to_2010_inpatient_claims_sample_{s}.zip",
    f"outpatient_s{s}": cms + f"de1_0_2008_to_2010_outpatient_claims_sample_{s}.zip",
    f"carrier_a_s{s}":  dl + f"DE1_0_2008_to_2010_Carrier_Claims_Sample_{s}A.zip",
    f"carrier_b_s{s}":  dl + f"DE1_0_2008_to_2010_Carrier_Claims_Sample_{s}B.zip",
    f"rx_s{s}":         dl + f"DE1_0_2008_to_2010_Prescription_Drug_Events_Sample_{s}.zip",
}

In [14]:
def download(name, url):
    out = RAW / f"{name}.zip"
    if out.exists():
        print("skip", name); return
    r = requests.get(url, stream=True, timeout=300)
    r.raise_for_status()
    with open(out, "wb") as f:
        for chunk in r.iter_content(1 << 20):
            f.write(chunk)
    print("ok", name, round(out.stat().st_size / 1e6, 1), "MB")

for name, url in URLS.items():
    download(name, url)

ok bene_2008_s2 3.1 MB
ok bene_2009_s2 3.2 MB
ok bene_2010_s2 3.0 MB
ok inpatient_s2 4.1 MB
ok outpatient_s2 34.5 MB
ok carrier_a_s2 113.4 MB
ok carrier_b_s2 113.5 MB
ok rx_s2 104.6 MB


In [15]:
for z in RAW.glob("*.zip"):
    if zipfile.is_zipfile(z):
        zipfile.ZipFile(z).extractall(RAW)
    else:
        print("NOT a zip (bad link?):", z.name)

sorted(p.name for p in RAW.glob("*.csv"))

['DE1_0_2008_Beneficiary_Summary_File_Sample_2.csv',
 'DE1_0_2008_to_2010_Carrier_Claims_Sample_2A.csv',
 'DE1_0_2008_to_2010_Carrier_Claims_Sample_2B.csv',
 'DE1_0_2008_to_2010_Inpatient_Claims_Sample_2.csv',
 'DE1_0_2008_to_2010_Outpatient_Claims_Sample_2.csv',
 'DE1_0_2008_to_2010_Prescription_Drug_Events_Sample_2.csv',
 'DE1_0_2009_Beneficiary_Summary_File_Sample_2.csv',
 'DE1_0_2010_Beneficiary_Summary_File_Sample_2.csv']

In [17]:
import pandas as pd

ben = pd.read_csv(next(RAW.glob("*2008_Beneficiary*.csv")), nrows=5)
ip  = pd.read_csv(next(RAW.glob("*Inpatient*.csv")), nrows=5)
car = pd.read_csv(next(RAW.glob("*Carrier_Claims_Sample_2A*.csv")), nrows=5)

print(ben.columns.tolist())
ip[["DESYNPUF_ID", "CLM_ID", "CLM_FROM_DT", "CLM_PMT_AMT",
    "CLM_DRG_CD", "ICD9_DGNS_CD_1", "ICD9_PRCDR_CD_1"]]

['DESYNPUF_ID', 'BENE_BIRTH_DT', 'BENE_DEATH_DT', 'BENE_SEX_IDENT_CD', 'BENE_RACE_CD', 'BENE_ESRD_IND', 'SP_STATE_CODE', 'BENE_COUNTY_CD', 'BENE_HI_CVRAGE_TOT_MONS', 'BENE_SMI_CVRAGE_TOT_MONS', 'BENE_HMO_CVRAGE_TOT_MONS', 'PLAN_CVRG_MOS_NUM', 'SP_ALZHDMTA', 'SP_CHF', 'SP_CHRNKIDN', 'SP_CNCR', 'SP_COPD', 'SP_DEPRESSN', 'SP_DIABETES', 'SP_ISCHMCHT', 'SP_OSTEOPRS', 'SP_RA_OA', 'SP_STRKETIA', 'MEDREIMB_IP', 'BENRES_IP', 'PPPYMT_IP', 'MEDREIMB_OP', 'BENRES_OP', 'PPPYMT_OP', 'MEDREIMB_CAR', 'BENRES_CAR', 'PPPYMT_CAR']


,DESYNPUF_ID,CLM_ID,CLM_FROM_DT,CLM_PMT_AMT,CLM_DRG_CD,ICD9_DGNS_CD_1,ICD9_PRCDR_CD_1
0,00000B48BCF4AD29,45461150060934,20080707,57000.0,281,4241,3521.0
1,00000B48BCF4AD29,45541150055035,20080724,0.0,296,42823,NaN
2,00000B48BCF4AD29,45731150068478,20080805,24000.0,263,3970,3615.0
3,0000525AB30E4DEF,45811150057351,20080525,4000.0,349,5609,9915.0
4,0000525AB30E4DEF,45391150058019,20080918,8000.0,388,5579,3898.0


In [18]:
car[["DESYNPUF_ID", "CLM_ID", "ICD9_DGNS_CD_1", "HCPCS_CD_1", "LINE_NCH_PMT_AMT_1"]]

,DESYNPUF_ID,CLM_ID,ICD9_DGNS_CD_1,HCPCS_CD_1,LINE_NCH_PMT_AMT_1
0,00000B48BCF4AD29,737133358637027,7061,99213,70.0
1,00000B48BCF4AD29,737783362300937,9561,73550,280.0
2,00000B48BCF4AD29,737513360019665,8241,99214,150.0
3,00000B48BCF4AD29,737553359102154,V676,80100,110.0
4,00000B48BCF4AD29,737033360678635,5765,76705,30.0
